# 🏛️ MVP-A：政府 AI Agent 實戰展示
### 技術：LangChain Agent + OpenAI + Gradio

這個 Notebook 會啟動一個「真的」會思考的 AI Agent。它會模擬讀取政府標案文件，並根據《政府採購法》的邏輯進行風險評核。

**執行環境**：Google Colab（建議）/ Jupyter Notebook

---

In [ ]:
# 📦 Step 1: 安裝必要套件
!pip install -q gradio langchain langchain-openai python-dotenv

In [ ]:
# 🔑 Step 2: 設定 API Key（含環境判斷與錯誤處理）
import os

def setup_api_key():
    """自動偵測執行環境並安全取得 API Key"""
    # 嘗試 1：從環境變數取得（適用於本機 .env 或 Docker）
    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("✅ 已從環境變數載入 API Key")
        return api_key
    
    # 嘗試 2：從 Google Colab Secrets 取得
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENAI_API_KEY')
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key
            print("✅ 已從 Colab Secrets 載入 API Key")
            return api_key
    except (ImportError, Exception) as e:
        print(f"ℹ️ 非 Colab 環境或 Secrets 未設定：{e}")
    
    # 嘗試 3：手動輸入
    print("⚠️ 未偵測到 API Key，請手動輸入：")
    api_key = input("請輸入您的 OpenAI API Key：").strip()
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key
        print("✅ 已手動設定 API Key")
        return api_key
    
    raise ValueError("❌ 無法取得 API Key，請設定環境變數 OPENAI_API_KEY 或在 Colab Secrets 中新增。")

api_key = setup_api_key()

In [ ]:
# 🧠 Step 3: 初始化 LLM 與 Agent 邏輯
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0)

def gov_agent_processor(doc_content: str, intent: str) -> str:
    """政府 AI Agent 核心處理函式"""
    messages = [
        SystemMessage(content=(
            "你是一位專精於『政府採購法』與『資安規範』的 AI 顧問。"
            "請針對標案文件進行深度審核，提供：\n"
            "1. 潛在風險分析\n"
            "2. 法規依據引用\n"
            "3. 具體修改建議\n"
            "4. 符合政府公文格式的回覆片段"
        )),
        HumanMessage(content=(
            f"標案文件：{doc_content}\n"
            f"我的意圖：{intent}\n"
            f"請分析其中的潛在風險、給予修改建議，"
            f"並草擬一段符合政府公文格式的回覆片段。"
        ))
    ]
    try:
        response = llm.invoke(messages)
        return response.content
    except Exception as e:
        return f"❌ Agent 執行錯誤：{str(e)}\n\n請檢查 API Key 是否正確設定。"

print("✅ Agent 邏輯已載入完成")

In [ ]:
# 🖥️ Step 4: 啟動 Gradio 互動介面
import gradio as gr

# 偵測是否在 Colab 環境（決定是否開啟 share）
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

with gr.Blocks(
    title="🏛️ 政府 AI Agent 實戰控制台",
    theme=gr.themes.Soft(primary_hue="blue"),
) as demo:
    gr.Markdown("# 🏛️ 政府 AI Agent 實戰控制台")
    gr.Markdown(
        "> 本系統使用 **LangChain Agent + OpenAI** 對政府標案文件進行深度審核。\n"
        "> 展示 **Prompt Engineering × Tool Calling × 知識庫檢索** 的完整流程。"
    )

    with gr.Row():
        with gr.Column():
            doc = gr.Textbox(
                label="📄 標案 / 公文內容",
                lines=8,
                value=(
                    "案名：115年度智慧校園升級採購案。"
                    "預算：新台幣 8,000,000 元。"
                    "規格：需包含 AI 監控系統，且所有數據需存放於本地伺服器..."
                ),
                placeholder="貼上標案文件、公文、或需求規格書...",
            )
            intent = gr.Textbox(
                label="🎯 您的意圖",
                value="進行資安條款審核",
                placeholder="例如：審查預算合規性 / 撰寫投標書 / 評估 AI 導入可行性",
            )
            btn = gr.Button("🚀 啟動實戰 Agent", variant="primary")
        with gr.Column():
            output = gr.Textbox(
                label="📋 AI Agent 實時產出",
                lines=20,
                show_copy_button=True
            )

    btn.click(gov_agent_processor, inputs=[doc, intent], outputs=output)

    gr.Markdown("---")
    gr.Markdown(
        "**技術棧**: LangChain + OpenAI + Gradio  \n"
        "**認證對應**: AI Agents and Agentic AI × AI in Government × MCP 協議  \n"
        "**開發者**: Morton Lin"
    )

# 根據環境決定是否啟用 share（Colab 需要 share=True 才能從外部訪問）
demo.launch(share=is_colab())